# Notebook 2v2 — Document Manipulation (Updated SEO Prompts)

Independent second run with updated SEO and SEOGEO prompts (Niklas revision).
GEO is reused from the first run — only SEO and SEOGEO are re-generated.

**Input/Output:** `data/{domain}/selected_docs_v2.json`

## Instructions
1. Run **Setup**
2. Run **Define Updated Methods**
3. Run **Create selected_docs_v2.json** — copies GEO from first run, strips SEO and SEOGEO
4. Run **Parameters**
5. Run **Manipulation**
6. Run **Inspect Results** to verify quality

## Setup

In [1]:
import json
import os
import sys
import time
from datetime import datetime

notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, ".."))
sys.path.insert(0, os.path.join(project_root, "src"))

from llms import OpenAIHelper
from llms.llm_interface import LLMInterface
from methods.citation_boosting import CitationBoosting
from methods import GEOOptimization

# Load API key
config_path = os.path.join(project_root, "config.json")
with open(config_path, "r") as f:
    config = json.load(f)
os.environ["OPENAI_API_KEY"] = config["OPENAI_API_KEY"]

data_dir = os.path.join(project_root, "data")

print("Setup complete.")
print(f"Project root: {project_root}")

Setup complete.
Project root: /Users/leonardrampf/Library/CloudStorage/OneDrive-Personal/Dokumente/Universität/Nova SBE/Work Project/seo-geo-generative-search-experiment


## Define Updated Methods

New SEO and SEOGEO prompts defined inline — does not affect `optimization_methods.py`.

In [2]:
class SEOOptimizationV2(CitationBoosting):
    """
    Updated SEO prompt (Niklas revision).
    Edits: Markdown Headings, Vocabulary Enrichment, Q&A Query Symmetry.
    """
    def __init__(self, llm: LLMInterface):
        super().__init__(llm)
        self.system_prompt = """You are an expert ml researcher having previous background in SEO and information retrieval. You are working on novel research ideas for next generation search systems, specifically how documents can be optimized to rank higher in modern semantic search engines such as Vertex AI Search. These systems use a combination of lexical matching, semantic understanding, and neural retrieval to rank documents. Your task is to revise source documents by applying targeted on-page SEO edits that improve how these retrieval systems understand and rank the content."""
        self.prompt_template = """Revise the following source to improve its ranking in modern semantic search engines. Apply the following three edits:

1. Semantic Structure and Headings: Logically break up the text to create a clear semantic hierarchy. You MUST use standard Markdown for headings (e.g., '##' or '###') to separate main topics and subtopics. Do not use raw HTML.

2. Keyword Context and Natural Language: Enrich the existing text by naturally incorporating synonyms, related terms, and variations of the core topic. Do not broaden the scope into a general summary; maintain the specific focus of the original text while using a richer vocabulary. The revised document must stay within +/- 10% of the original word count.

3. Q&A / Query Symmetry: Transform one key concept within the text into a direct Question and Answer format. Explicitly pose a question that a user searching for this topic would likely ask, and immediately provide a dense, direct answer. Embed this naturally into the top half of the document. Do not bloat the document — delete original filler words if necessary to ensure the final output length stays within +/- 10% of the original word count.

Preserve all core facts and information from the original source. You are optimizing the delivery and structure, not changing the underlying truth of the content.

Source:
```
{description}
```

Do not explain the changes, just give me the revised source."""

    def post_processing(self, text: str) -> str:
        if "Updated Source:" in text:
            text = text.split("Updated Source:")[-1]
        text = text.strip().strip("```").strip()
        return text


class SEOGEOOptimizationV2(CitationBoosting):
    """
    Updated SEOGEO prompt (Niklas revision for SEO edits).
    GEO edits (1-3) unchanged. SEO edits (4-6) updated to Niklas version.
    """
    def __init__(self, llm: LLMInterface):
        super().__init__(llm)
        self.system_prompt = """You are an expert ml researcher having previous background in SEO and search engines in general. You are working on novel research ideas for next generation of products. These products will have language models augmented with search engines, with the task of answering questions based on sources backed by the search engine. This new set of systems will be collectively called language engines (generative search engines). Specifically they will use a combination of GEO (Generative Engine Optimization) techniques to boost their visibility in the final text answer outputted by the Language Engine, as well as traditional SEO techniques to rank higher in modern semantic search engines such as Vertex AI Search."""
        self.prompt_template = """Revise the following source to improve both its ranking in modern semantic search engines and its visibility when presented as a search result to a language model. Apply the following six edits:

1. Add relevant statistics and numerical facts at multiple places in the text. These can be hypothetical but must sound realistic and contextually appropriate. Add them inline within existing sentences — no separate paragraphs.

2. Add citations from credible sources in natural language. You may invent these sources but ensure they sound plausible. For example: "According to Google's latest report..." or "A study by Nielsen found that...". Around 4-5 citations in the whole source are enough provided they are relevant and the text looks natural. Do not use research paper style citations.

3. Rewrite the source to sound more confident, expert-like, and authoritative. Replace uncertain or weak phrasing ("might", "could", "possibly", "may be useful") with assertive, definitive language. Present statements as established facts rather than suggestions. The goal is that readers perceive this source as more credible and well-informed than competing sources.

4. Semantic Structure and Headings: Logically break up the text to create a clear semantic hierarchy. You MUST use standard Markdown for headings (e.g., '##' or '###') to separate main topics and subtopics. Do not use raw HTML.

5. Keyword Context and Natural Language: Enrich the existing text by naturally incorporating synonyms, related terms, and variations of the core topic. Do not broaden the scope into a general summary; maintain the specific focus of the original text while using a richer vocabulary. The revised document must stay within +/- 10% of the original word count.

6. Q&A / Query Symmetry: Transform one key concept within the text into a direct Question and Answer format. Explicitly pose a question that a user searching for this topic would likely ask, and immediately provide a dense, direct answer. Embed this naturally into the top half of the document. Do not bloat the document — delete original filler words if necessary to ensure the final output length stays within +/- 10% of the original word count.

Preserve all core facts and information from the original source. You are optimizing the delivery and structure, not changing the underlying truth of the content.

Source:
```
{description}
```

Do not explain the changes, just give me the revised source."""

    def post_processing(self, text: str) -> str:
        if "Updated Source:" in text:
            text = text.split("Updated Source:")[-1]
        text = text.strip().strip("```").strip()
        return text


print("SEOOptimizationV2 and SEOGEOOptimizationV2 defined.")

SEOOptimizationV2 and SEOGEOOptimizationV2 defined.


## Create selected_docs_v2.json

Copies `selected_docs.json` into `selected_docs_v2.json`.
Keeps `GEO(doc)` from the first run — strips `SEO(doc)` and `SEOGEO(doc)`.

**Only run this once.** If `selected_docs_v2.json` already exists, this cell is skipped.

In [3]:
DOMAINS_TO_PREPARE = ["retail"]  # <- match DOMAINS below

for domain in DOMAINS_TO_PREPARE:
    src_path = os.path.join(data_dir, domain, "selected_docs.json")
    dst_path = os.path.join(data_dir, domain, "selected_docs_v2.json")

    if os.path.exists(dst_path):
        print(f"[{domain}] selected_docs_v2.json already exists — skipping.")
        continue

    if not os.path.exists(src_path):
        print(f"[{domain}] WARNING: selected_docs.json not found. Run Notebook 1 first.")
        continue

    with open(src_path, "r", encoding="utf-8") as f:
        docs = json.load(f)

    # Keep GEO(doc) — strip SEO(doc) and SEOGEO(doc)
    for entry in docs.values():
        entry.pop("SEO(doc)", None)
        entry.pop("SEOGEO(doc)", None)

    geo_done = sum(1 for d in docs.values() if d.get("GEO(doc)") is not None)

    with open(dst_path, "w", encoding="utf-8") as f:
        json.dump(docs, f, indent=4, ensure_ascii=False)

    print(f"[{domain}] Created selected_docs_v2.json")
    print(f"  GEO(doc) preserved: {geo_done}/{len(docs)}")
    print(f"  SEO(doc) and SEOGEO(doc) removed — will be re-generated.")

[retail] Created selected_docs_v2.json
  GEO(doc) preserved: 500/500
  SEO(doc) and SEOGEO(doc) removed — will be re-generated.


## Parameters

In [4]:
# LLM for manipulation
LLM_NAME = "gpt-4o-mini-2024-07-18"

# <- CHANGE THIS to control which domains to process
DOMAINS = ["retail"]

# Methods to apply — GEO skipped automatically if already done
METHOD_NAMES = ["SEO", "GEO", "SEOGEO"]

# Method classes — updated V2 for SEO and SEOGEO, unchanged for GEO
method_classes = {
    "SEO":    SEOOptimizationV2,
    "GEO":    GEOOptimization,
    "SEOGEO": SEOGEOOptimizationV2,
}

# Calculate total remaining work
total_calls = 0
for domain in DOMAINS:
    path = os.path.join(data_dir, domain, "selected_docs_v2.json")
    if os.path.exists(path):
        with open(path, "r") as f:
            docs = json.load(f)
        remaining = sum(
            len(METHOD_NAMES) - sum(1 for m in METHOD_NAMES if docs[idx].get(f"{m}(doc)") is not None)
            for idx in docs
        )
        total_calls += remaining

print(f"LLM:             {LLM_NAME}")
print(f"Domains:         {DOMAINS}")
print(f"Methods:         {METHOD_NAMES}")
print(f"Remaining calls: {total_calls}")
print(f"Estimated time:  {total_calls * 4 // 60}-{total_calls * 5 // 60} minutes")

LLM:             gpt-4o-mini-2024-07-18
Domains:         ['retail']
Methods:         ['SEO', 'GEO', 'SEOGEO']
Remaining calls: 1000
Estimated time:  66-83 minutes


## Manipulation

Reads and writes `selected_docs_v2.json` — fully independent from the main run.
GEO is skipped automatically if already done. Safe to interrupt and resume.

In [7]:
llm = OpenAIHelper(LLM_NAME)
start_time = datetime.now()
print(f"Started at: {start_time.strftime('%H:%M:%S')}")
print()

for domain in DOMAINS:
    selected_docs_path = os.path.join(data_dir, domain, "selected_docs_v2.json")

    if not os.path.exists(selected_docs_path):
        print(f"[{domain}] WARNING: selected_docs_v2.json not found.")
        print(f"  Run the 'Create selected_docs_v2.json' cell first.")
        continue

    with open(selected_docs_path, "r", encoding="utf-8") as f:
        selected_docs = json.load(f)

    query_indices = list(selected_docs.keys())
    total_docs = len(query_indices)

    print(f"{'='*60}")
    print(f"DOMAIN: {domain.upper()} ({total_docs} documents)")
    print(f"{'='*60}")

    for method_name in METHOD_NAMES:
        method = method_classes[method_name](llm)
        already_done = sum(1 for d in selected_docs.values() if d.get(f"{method_name}(doc)") is not None)

        if already_done == total_docs:
            print(f"  [{method_name}] All {total_docs} docs already done — skipping")
            continue

        print(f"  [{method_name}] Starting ({already_done}/{total_docs} already done)")

        for i, query_idx in enumerate(query_indices):
            if selected_docs[query_idx].get(f"{method_name}(doc)") is not None:
                continue

            doc = selected_docs[query_idx]["doc"]

            try:
                result = method.improve_text(doc)
                selected_docs[query_idx][f"{method_name}(doc)"] = result
                with open(selected_docs_path, "w", encoding="utf-8") as f:
                    json.dump(selected_docs, f, indent=4, ensure_ascii=False)
                print(f"    [{i+1}/{total_docs}] {query_idx}: done")

            except Exception as e:
                print(f"    [{i+1}/{total_docs}] {query_idx}: ERROR — {e}")
                print(f"    Retrying in 10 seconds...")
                time.sleep(10)
                try:
                    result = method.improve_text(doc)
                    selected_docs[query_idx][f"{method_name}(doc)"] = result
                    with open(selected_docs_path, "w", encoding="utf-8") as f:
                        json.dump(selected_docs, f, indent=4, ensure_ascii=False)
                    print(f"    [{i+1}/{total_docs}] {query_idx}: done (retry OK)")
                except Exception as e2:
                    print(f"    [{i+1}/{total_docs}] {query_idx}: FAILED — {e2}")
                    selected_docs[query_idx][f"{method_name}(doc)"] = None

        done = sum(1 for d in selected_docs.values() if d.get(f"{method_name}(doc)") is not None)
        print(f"  [{method_name}] Complete: {done}/{total_docs}")

    print(f"  Domain '{domain}' done at {datetime.now().strftime('%H:%M:%S')}")
    print()

end_time = datetime.now()
elapsed = end_time - start_time
print(f"{'='*60}")
print(f"ALL DOMAINS COMPLETE")
print(f"Started:  {start_time.strftime('%H:%M:%S')}")
print(f"Finished: {end_time.strftime('%H:%M:%S')}")
print(f"Total time: {str(elapsed).split('.')[0]}")

Started at: 11:23:16

DOMAIN: RETAIL (500 documents)
  [SEO] All 500 docs already done — skipping
  [GEO] All 500 docs already done — skipping
  [SEOGEO] Starting (176/500 already done)
    [177/500] 176: done
    [178/500] 177: done
    [179/500] 178: done
    [180/500] 179: done
    [181/500] 180: done
    [182/500] 181: done
    [183/500] 182: done
    [184/500] 183: done
    [185/500] 184: done
    [186/500] 185: done
    [187/500] 186: done
    [188/500] 187: done
    [189/500] 188: done
    [190/500] 189: done
    [191/500] 190: done
    [192/500] 191: done
    [193/500] 192: done
    [194/500] 193: done
    [195/500] 194: done
    [196/500] 195: done
    [197/500] 196: done
    [198/500] 197: done
    [199/500] 198: done
    [200/500] 199: done
    [201/500] 200: done
    [202/500] 201: done
    [203/500] 202: done
    [204/500] 203: done
    [205/500] 204: done
    [206/500] 205: done
    [207/500] 206: done
    [208/500] 207: done
    [209/500] 208: done
    [210/500] 209: don

## Inspect Results

In [8]:
INSPECT_DOMAIN = "retail"
INSPECT_QUERY_IDX = "0"

inspect_path = os.path.join(data_dir, INSPECT_DOMAIN, "selected_docs_v2.json")
with open(inspect_path, "r", encoding="utf-8") as f:
    selected_docs = json.load(f)

example = selected_docs[INSPECT_QUERY_IDX]
print(f"Domain: {INSPECT_DOMAIN}")
print(f"Query: {example['query']}")

print(f"\n--- ORIGINAL ---")
print(example["doc"][:500])
print(f"\n--- SEO (V2) ---")
print(example.get("SEO(doc)", "NOT YET GENERATED")[:500])
print(f"\n--- GEO ---")
print(example.get("GEO(doc)", "NOT YET GENERATED")[:500])
print(f"\n--- SEOGEO (V2) ---")
print(example.get("SEOGEO(doc)", "NOT YET GENERATED")[:500])

Domain: retail
Query: holidaytraditions

--- ORIGINAL ---
Name: Kids Christmas Ornaments 2021 – Personalized Garbage Truck Ornaments for Christmas Tree – Garbage Truck Ornament – Fun Ornaments for Teens & Kids 2,3,4,5,6,7,8,9 - Garbage Truck Christmas Ornament
Description:
List of features:
WHEN IT COMES TO YOUR Personalized KIDS and TEEN ornaments, you want high-quality, BEAUTIFULLY crafted ornaments that will elevate your family’s holiday celebration and LAST THROUGH GENERATIONS – not flimsy Christmas hanging ornaments that break & FALL APART in a ma

--- SEO (V2) ---
# Personalized Kids Christmas Ornaments 2021

## Overview
**What makes Personalized Kids Christmas Ornaments special?**  
These festive ornaments, featuring adorable garbage truck designs, are crafted with high-quality materials, ensuring they’re not just beautiful decorations but lasting keepsakes for your family.

## Key Features

### Quality Craftsmanship
When selecting personalized ornaments for your kids and teens,